In [2]:
!pip install groq python-dotenv numpy tqdm datasets

In [3]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

In [4]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama-3.1-8b-instant"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.3, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content
    
    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [5]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello world. I'm here to help with any math problems you might have. What's on your mind? Do you have a specific problem you'd like me to solve, or would you like some help with a particular math concept?


#### GSM8K 데이터셋 확인해보기

In [6]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [12]:
### 수정해도 됩니다!
def extract_final_answer(response: str):
    if not response:
        return None
    
    m = re.search(r"Answer:\s*\$?\s*(-?\d[\d,]*(?:\.\d+)?)\b", response)
    if m:
        return m.group(1).replace(",", "")
    
    m = re.search(r"Model response:\s*\$?\s*(-?\d[\d,]*(?:\.\d+)?)\b", response)
    if m:
        return m.group(1).replace(",", "")

    m = re.search(r"(-?\d[\d,]*(?:\.\d+)?)\s*(meters|cups|miles|minutes)\b", response)
    if m:
        return m.group(1).replace(",", "")

    nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", response)
    return nums[-1].replace(",", "") if nums else None

In [8]:
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        ans_text = dataset[i]["answer"].split("####")[-1].strip()
        m = re.search(r"-?\d[\d,]*(?:\.\d+)?", ans_text)
        correct_answer = float(m.group().replace(",", "")) if m else None

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        total += 1

        if VERBOSE:
            print("="*50)
            print(response)
            print("="*50)

        predicted_answer_raw = extract_final_answer(response) if response else None
        predicted_answer = None

        if isinstance(predicted_answer_raw, str):
            predicted_answer_raw = predicted_answer_raw.strip()
            if predicted_answer_raw != "":
                try:
                    predicted_answer = float(predicted_answer_raw.replace(",", ""))
                except ValueError:
                    m = re.search(r"-?\d[\d,]*(?:\.\d+)?", predicted_answer_raw)
                    if m:
                        predicted_answer = float(m.group())

        is_correct = False
        if (predicted_answer is not None) and (correct_answer is not None):
            diff = abs(predicted_answer - correct_answer)
            is_correct = diff < 1e-5
        else:
            is_correct = False

        if is_correct:
            correct += 1
            
        results.append({
            'question': question,
            'correct_answer': correct_answer,
            'predicted_answer': predicted_answer,
            'response': response,
            'correct': is_correct
        })

        if (i + 1) % 5 == 0:
            current_acc = correct/total if total > 0 else 0
            print(f"Progress: [{i+1}/{num_samples}]")
            print(f"Current Acc.: [{current_acc:.2%}]")
        
    return results, correct/total if total > 0 else 0

In [13]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"
    
    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [38]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [17]:
example_list = get_few_shot_examples()
PROMPT = construct_my_prompt(example_list, num_examples=5)
VERBOSE = False

results, accuracy = run_benchmark_test_efficient(
    dataset=gsm8k_test,
    prompt_template=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=50
)
save_final_result(results, accuracy, "My_prompting_5.txt")

 10%|█         | 5/50 [00:12<01:53,  2.53s/it]


Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:24<01:38,  2.45s/it]


Progress: [10/50]
Current Acc.: [70.00%]


 30%|███       | 15/50 [00:43<02:28,  4.24s/it]

...


Progress: [40/50]
Current Acc.: [85.00%]


 90%|█████████ | 45/50 [03:35<00:29,  5.88s/it]


Progress: [45/50]
Current Acc.: [82.22%]


100%|██████████| 50/50 [04:05<00:00,  4.91s/it]


Progress: [50/50]
Current Acc.: [82.00%]


In [ ]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!

### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [10]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )
    prompt = (
        "Instruction:\n"
        "Solve the following math problem step by step.\n"
        "Your response MUST end with a final line exactly:\n"
        "Answer: <number>\n"
        "The Answer line must contain ONLY digits (and an optional decimal point). "
        "No commas, no currency symbols, no units, no extra words.\n"
        ) #TODO: 프롬프트를 작성해주세요!

    for i in range(num_examples):
        #TODO: CoT example을 만들어주세요!
        index = sampled_indices[i]
        cur_question = train_dataset['question'][index]
        cur_solution = train_dataset['answer'][index].strip()
        cur_final = cur_solution.split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += "Reasoning:\n"
        reasoning = cur_solution.split("####")[0].strip()
        prompt += f"{reasoning}\n"
        prompt += f"Answer: {cur_final}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [ ]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!

### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [14]:
### 자유롭게 수정해도 됩니다! 완전히 새로 함수를 만들어도 돼요.
from typing import List
from collections import Counter
import time
import re
from tqdm import tqdm


def generate_response_for_my_strategy(
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        temperature: float = 0.0,
        max_tokens: int = 512,
        stop: list = None
    ):
  
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a helpful assistant that solves math problems."},
                {"role": "user", "content": prompt}
            ],
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            stream=False
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        print(f"API call error: {str(e)}")
        time.sleep(5) 
        return None

def run_benchmark_test_efficient(
        dataset,
        prompt_template: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False,
        log_every: int = 5,              
        show_example: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        ans_text = dataset[i]["answer"].split("####")[-1].strip()
        m = re.search(r"-?\d[\d,]*(?:\.\d+)?", ans_text)
        correct_answer = float(m.group().replace(",", "")) if m else None

        time.sleep(2.0) 

        response = generate_response_for_my_strategy(
            prompt=prompt_template.format(question=question),
            model=model,
            temperature=0.0 
        )
        
        predicted_answer = None
        if response:
            pred_str = extract_final_answer(response)
            if pred_str:
                try:
                    predicted_answer = float(pred_str.replace(",", ""))
                except:
                    pass
        
        total += 1
        is_correct = False
        if (predicted_answer is not None) and (correct_answer is not None):
            diff = abs(predicted_answer - correct_answer)
            is_correct = diff < 1e-5
        
        if is_correct:
            correct += 1
            
        results.append({
            'question': question,
            'correct_answer': correct_answer,
            'predicted_answer': predicted_answer,
            'response': response,
            'correct': is_correct
        })

        if log_every > 0 and (i + 1) % log_every == 0:
            current_acc = correct / total if total > 0 else 0
            print(f"\nProgress: [{i+1}/{num_samples}]")
            print(f"Current Acc.: [{current_acc:.2%}]")

            if show_example:
                print("-" * 80)
                print("[Question]")
                print(question)
                print("[GT Answer]", correct_answer)
                print("[Predicted]", predicted_answer)
                if VERBOSE and response:
                    print("[Model Response]")
                    print(response)
                print("-" * 80)
        
    final_acc = correct / total if total > 0 else 0
    return results, final_acc

def get_few_shot_examples() -> List[str]:
    ex1 = (
        "Question:\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. "
        "How many clips did Natalia sell altogether in April and May?\n"
        "Reasoning:\nApril=48. May=48/2=24. Total=48+24=72.\n"
        "Answer: 72"
    )

    ex2 = (
        "Question:\nWeng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. "
        "How much did she earn?\n"
        "Reasoning:\n$12 per 60 min => 12/60=0.2 per min. 0.2*50=10.\n"
        "Answer: 10"
    )

    ex3 = (
        "Question:\nBetty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. "
        "Her parents give her $15. How much money does she still need?\n"
        "Reasoning:\nHalf of 100 is 50. 50+15=65. Need 100-65=35.\n"
        "Answer: 35"
    )

    ex4 = (
        "Question:\nJulie is reading a 120-page book. Yesterday 12 pages, today twice that. "
        "If she reads half of the remaining pages tomorrow, how many pages is that?\n"
        "Reasoning:\nToday=24. Read=12+24=36. Remaining=120-36=84. Half=42.\n"
        "Answer: 42"
    )

    ex5 = (
        "Question:\nJames runs 3 sprints 3 times a week. Each sprint is 60 meters. How many meters per week?\n"
        "Reasoning:\nTotal sprints=3*3=9. Distance=9*60=540.\n"
        "Answer: 540"
    )

    return [ex1, ex2, ex3, ex4, ex5]


def construct_my_prompt(example_list: List[str], num_examples: int = 3):
    instruction = (
        "Instruction:\n"
        "Solve the math problem step by step.\n"
        "At the end, write the final answer on the LAST line exactly in this format:\n"
        "Answer: <number>\n"
        "The Answer line must contain ONLY digits (and an optional decimal point).\n"
        "No commas, no units, no extra words.\n"
    )

    prompt = instruction + "\n"

    for ex in example_list[:num_examples]:
        prompt += ex.strip() + "\n\n"

    prompt += (
        "Question:\n{question}\n"
        "Reasoning:\n"
        "Answer:"
    )
    return prompt

In [ ]:
# TODO: 만든 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!

### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!

## 1. Prompting Strategy별 정답률 비교

| Prompting Method | 0-shot | 3-shot | 5-shot |
|------------------|--------|--------|--------|
| Direct Prompting | 0.78   | 0.68   | 0.80   |
| CoT Prompting    | 0.82   | 0.84   | 0.80   |
| My Prompting     | 0.86   | 0.86   | 0.82   |

표를 통해 확인할 수 있듯이, prompting 전략에 따라 few-shot 예제 증가에 대한 반응이 다르게 나타난다.
Direct Prompting의 경우 3-shot에서 오히려 성능이 하락하는 현상을 보였는데 이는 예제가 추가되었음에도 불구하고 명시적인 추론 지시가 없어 모델이 예제를 효과적으로 일반화하지 못했기 때문으로 해석할 수 있다.
반면 CoT Prompting과 My Prompting은 few-shot 환경에서 비교적 안정적인 성능을 유지하며 특히 구조화된 추론을 요구하는 프롬프트가 예제의 도움을 더 잘 활용함을 보여준다.

한편 5-shot 환경에서는 모든 prompting 전략에서 성능 향상이 제한적으로 나타났다. 이는 GSM8K 문제의 특성상 일정 수준 이상의 예제가 제공되면 추가 예제가 더 이상 큰 정보를 제공하지 못하고 오히려 입력 길이 증가로 인해 효율이 감소할 수 있음을 시사한다.


## 2. CoT Prompting이 Direct Prompting보다 효과적인 이유

Direct Prompting은 모델이 문제를 보고 곧바로 정답을 출력하도록 유도하는 방식이다. 이 방식은 간단한 문제에서는 효율적일 수 있지만, 여러 단계의 계산이나 조건 해석이 필요한 문제에서는 중간 추론 과정이 생략되어 계산 실수나 논리적 오류가 발생할 가능성이 높다.

반면, CoT Prompting은 문제를 단계별로 reasoning하도록 유도함으로써 모델이 문제의 조건을 명확히 파악하고 중간 계산을 체계적으로 수행할 수 있게 한다.

이로 인해 복합적인 수학 문제에서 정답률이 향상될 수 있으며 실험 결과에서도 CoT Prompting은 Direct Prompting보다 전반적으로 더 높은 성능을 보였다.

## 3. My Prompting이 CoT Prompting보다 효과적인 이유

직접 설계한 My Prompting은 CoT Prompting의 "단계적 추론 유도"라는 장점을 유지하면서 추가적으로 **출력 형식의 일관성**과 **평가/파싱 안정성**을 강화하여 성능이 더 높아질 수 있도록 하였다.

My Prompting에서는 문제 해결 과정을 단계적으로 수행한 뒤 마지막 줄에 반드시 `Answer: <number>` 형식으로 정답을 출력하도록 강제하였다. 이러한 구조는 모델의 추론을 작성한 뒤 자연스럽게 정답 라인을 완성하도록 유도한다. 이를 통해 정답 추출 함수가 "Answer: " 패턴을 안정적으로 매칭할 수 있다. 이는 CoT Prompting에서 발생할 수 있는 정답 라인이 애매하거나 문장으로 섞여 나올 수 있는 문제를 줄이고 정답 파싱 실패로 인한 오답 처리 가능성을 감소시킨다.

또한 few-shot 예제를 통해 Question, Reasoning, Answer의 흐름을 짧고 일관된 형식으로 학습시킴으로써 모델이 안정적으로 동일한 추론 패턴을 따르도록 유도하였다. 예제 Reasoning은 계산 흐름을 압축된 형태로 제공한다. 이를 통해 모델이 불필요하게 긴 서술을 생성하는 것을 줄이고 계산 단계를 명확히 따라가게 만들어 계산 실수를 줄이는 데 도움이 될 수 있다.

재현성 높은 추론 경로를 선택하기 위하여 temperature=0.0으로 고정하였다. 이를 통해 결과의 변동성을 줄여 실험의 재현성을 높일 수 있고 self-consistency처럼 다중 샘플링을 하지 않더라도 안정적인 성능을 얻을 수 있다.